# 07_00 Advanced: the Iliad, and combining both (overview)

*📖 Not in the book*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → model learns → model writes → MCP tool
> ```
>
> You are here: **every stage again**, on a second poem
>
> Project files this connects to:
>
> - `data/homer/`
> - `src/homer_ai/train_homer.py`
>
> **This notebook:** We have an Odyssey model. What do we gain by doing it all again on the Iliad, how do we compare the two fairly, and how could we combine them?
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- Why we repeat every stage on the Iliad, and what that proves
- What it means to compare two models fairly
- Three ways to combine two poems, and which ones this chapter builds
- Which notebook to open for each idea

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **Iliad** | Homer's other poem: 24 books about the anger of Achilles in the last year of the Trojan War | its first word, `μῆνιν`, means "anger" |
| **shared vocabulary** | one tokenizer, built from all the texts a comparison uses, so every model uses the same numbers for the same letters | `α` is number 32 for every model |
| **fine-tuning** | taking a model already trained on one text and training it a little more on another | an Odyssey model, then more training on the Iliad |

## 3. The idea

### ❓ We have an Odyssey model. Why build a second one?

Say you've trained a tour guide for one city, and now you need one for a second city. If the same lessons, unchanged, turn out a second good guide, then the lessons work and the first guide wasn't a lucky accident. With two guides you can also be fair about who knows what: ask both the same questions about both cities and compare. And then there's the question this chapter ends on: would one guide who studied both cities do as well as the two specialists, or do you keep both and send each visitor to the right one? The picture promises too much in one way: a guide knows facts about a city, and our models know only which letter tends to follow which, so "knowing a poem" here means a slightly lower loss on it.

Now in our own terms, where the lessons are our code and the cities are the two poems. First, as proof. If the same code, unchanged, learns a second poem, then the code is general and not a trick that only works on Book 1. Second, to compare. With two models we can ask real questions, such as "does a model know its own poem better than the other one?". And then we can combine them, into one tool that knows both poems.

```
Odyssey Greek ──▶ [ the same steps as chapters 2-5 ] ──▶ Odyssey model
Iliad Greek   ──▶ [ the same steps as chapters 2-5 ] ──▶ Iliad model
```

### ❓ If the code is the same, what actually changes?

Only the input: **which folder of text we read**. Everything after it (the tokenizer, the 90/10 split, the model, the training loop) is the code you already know. That's why `load_books` takes a folder as an argument. Here's that one change on the real texts, the same few lines run on two folders.

👉 **Predict first:** the two poems have different lengths. Will their vocabularies (the number of different characters, plus our 2 special tokens) have the same size?

In [ ]:
# data/homer: load_book_files reads all the books of a folder
from homer_ai.lessons import HOMER, load_book_files

In [ ]:
from homer_ai.tokenizer import CharTokenizer

for poem in ("odyssey", "iliad"):
    books = load_book_files(HOMER / poem / "greek")  # the same code for both poems
    text = "\n".join(books)
    tokenizer = CharTokenizer(text)
    print(
        f"{poem:8} {len(books)} books, {sum(len(b) for b in books):,} characters, "
        f"vocabulary {tokenizer.vocab_size}"
    )

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Two poems, one set of lines. The Iliad is longer, and its vocabulary is slightly bigger (171, against 169), because it uses a few characters the Odyssey never does. That small difference matters more than it looks, and [07_03](07_03_comparing_fairly.ipynb) shows why.

</details>

### ❓ Earlier our vocabulary was 137. Why are these numbers different?

137 was the tokenizer of chapters 1-6, built from Odyssey **Book 1 alone** (135 characters plus our 2 special tokens). In this chapter we read more text, so we meet more different characters. That gives 169 for the whole Greek Odyssey, 171 for the Iliad, 175 when one shared vocabulary is built from both, and 66 for Butler's English Book 1 (69 for the whole English poem). A token's number is its position in the sorted list, so a new character moves every letter after it. `α` is 26 in Book 1's tokenizer ([02_05](../02_text_to_numbers/02_05_token_embeddings.ipynb)), 31 in the Odyssey's and the Iliad's, and 32 in the shared one. Same letter, different number. That's the trap [07_03](07_03_comparing_fairly.ipynb) deals with.

### ❓ What does "compare two models fairly" mean?

Two models are only comparable if the test is the same for both. We use the habits of [05_04](../05_training_on_the_odyssey/05_04_running_fair_experiments.ipynb) (fix the seed, change one thing) and the held-out text of [05_05](../05_training_on_the_odyssey/05_05_validation.ipynb) (a part the model never trained on). One thing is new here: both models must use the same tokenizer, a **shared vocabulary**. Then we test each model on both poems:

```
                       tested on          tested on
                       Odyssey (held out) Iliad (held out)
trained on Odyssey          ?                  ?
trained on Iliad            ?                  ?
```

So the question is, does the model trained on a poem do better on that poem than the other model does? [07_03](07_03_comparing_fairly.ipynb) shows how to read the table.

### ❓ How can I combine two poems?

There are three ways. This chapter builds the first one.

```
1. ONE model, both texts     Odyssey ⏎ <|endoftext|> ⏎ Iliad  ──▶ one model   (07_04)
2. TWO models, side by side  Odyssey ──▶ model A   Iliad ──▶ model B
                             a program (like the MCP server) asks the one you need
3. FINE-TUNING               train on the Odyssey, then train a little more on the Iliad
```

Way 1 uses the `<|endoftext|>` marker of [02_02](../02_text_to_numbers/02_02_special_tokens.ipynb) between the poems. Way 2 needs no new maths. It's two copies of what you built, and [06_03](../06_making_it_talk/06_03_the_mcp_server.ipynb) shows how a server offers a model. Way 3, fine-tuning, is how big models are often adapted. This chapter doesn't build it for the two poems; [chapter 8](../08_advanced_fine_tuning/08_00_chapter_overview.ipynb) fine-tunes a much bigger model, GPT-2.

### ❓ Will the Iliad model write good Greek?

No, and I'd rather say so up front. In these notebooks our model is tiny (a few thousand characters of practice, a few seconds of training), so its writing is word-shaped pieces, like the Odyssey model in [06_02](../06_making_it_talk/06_02_less_repetition.ipynb). What we measure are **small differences between losses**, and we report them as they come, even when they're mixed.

Every cell here runs in seconds, so the training uses small slices of text and a few epochs. Full runs take much longer. [07_02](07_02_an_iliad_model.ipynb) gives the exact code to run them yourself.

### ❓ Which notebook do I open for each idea?

| Notebook | Question it answers |
|---|---|
| [`07_01_the_corpus.ipynb`](07_01_the_corpus.ipynb) | What's in `data/homer/`, in real numbers? |
| [`07_02_an_iliad_model.ipynb`](07_02_an_iliad_model.ipynb) | How do I train an Iliad model with the same code? |
| [`07_03_comparing_fairly.ipynb`](07_03_comparing_fairly.ipynb) | Why must two models share one vocabulary, and how do they compare? |
| [`07_04_combining_both.ipynb`](07_04_combining_both.ipynb) | What do I get, and lose, from one model on both poems? |
| [`07_05_english_and_greek.ipynb`](07_05_english_and_greek.ipynb) | Does the same code learn Butler's English? How does it differ from Greek? |

## 4. Recap

- The same steps on a new poem only change the folder we read. That shows the code is general.
- A fair comparison needs the same tokenizer (a shared vocabulary), the same seed, one change at a time, and held-out text.
- To combine poems you can train one model with `<|endoftext|>` between the works (built here), keep two models side by side, or fine-tune (chapter 8 does that, on GPT-2).

**Next:** [07_01: The corpus](07_01_the_corpus.ipynb).